# Proton Flux Spectrum (dN/dE via PSTAR)

Converts the Pockels-corrected pulse into an actual proton count per energy
bin, using PSTAR stopping power for the diamond detector -- see
`shot_characterization/flux.py` for the full physics chain and where every
constant comes from (recovered from the original `spectrum.ipynb`).

Requires both a Pockels correction (a logged EOM bias voltage) and a known
flight distance `L` (for the energy axis) -- only 4 of the 6
Pockels-correctable shots have both: 19, 20, 21, 27. Shots 9 and 11 have the
correction but no known `L` yet.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import shot_log as sl
from helpers import load_channel
from shot_characterization import (characterize, find_xray_shape, find_proton_onset,
                                    correct_pockels_signal, proton_flux_spectrum,
                                    proton_flux_total, flag_jitter_outliers)
from shot_characterization.characterize import regions_above

c = 299_792_458.0
m_p = 1.67262192369e-27
q_e = 1.602176634e-19
L = 3.12  # meters, source-to-detector distance (confirmed for shots 15+)


## Full pipeline, one shot at a time

Every stage from raw signal to flux spectrum, so each step's effect is
visible rather than hidden inside one function. Shot 19 first -- it turns
out to be the cleanest of the four (lowest exclusion, single-peak spectrum
shape), found by comparing all four after the fact.

In [ ]:
def compute_flux(shot, V_bias, L=L):
    entry = sl.main_channel_entry(shot)
    kind, path, ch = entry
    t, v, clipped, ydisp = load_channel(sl.DATA_DIR, kind, path, ch)
    t_ns = t * 1e9

    res = characterize(t, v, clipped)
    xr = find_xray_shape(t, v, res["baseline_V"], res["noise_V"])
    pr = find_proton_onset(t, v, res["baseline_V"], xr["peak_i"])

    result = correct_pockels_signal(t, v, V_bias=V_bias, V_baseline=res["baseline_V"])
    v_c = result["v_signal"]

    # two separate, independently-motivated exclusion criteria, combined:
    # pockels.py's own T-based mask (literally/near invalid transmission),
    # and a trend-deviation jitter check (noise riding on an otherwise-real
    # trend -- see flux.py docstring for why these aren't the same thing).
    # Checked directly: within the pulse window, the T-based mask alone
    # never independently flags anything the jitter check doesn't already
    # catch (0.0% "T-based only" on every shot) -- it's tuned for timing,
    # and the jitter check is doing essentially all the real protective
    # work here.
    baseline_mask = t_ns < 90
    jitter = flag_jitter_outliers(v_c, t_ns, baseline_mask, n_sigma=5.0)
    t_based = result["unstable_mask"]
    unreliable = t_based | jitter

    mask = (t_ns >= pr["onset_time_ns"]) & (t_ns <= pr["end_time_ns"])
    t_pulse = t_ns[mask]
    v_pulse = v_c[mask]
    unstable_pulse = unreliable[mask]

    L_over_c_ns = (L / c) * 1e9
    tof_pulse_s = (t_pulse - xr["onset_time_ns"] + L_over_c_ns) * 1e-9
    v_proton = L / tof_pulse_s
    beta = v_proton / c
    gamma = 1 / np.sqrt(1 - beta**2)
    energy_pulse_mev = (gamma - 1) * m_p * c**2 / q_e / 1e6

    dN_dE, order = proton_flux_spectrum(t_pulse, v_pulse, energy_pulse_mev,
                                         unstable_mask=unstable_pulse)
    total, frac_excluded = proton_flux_total(dN_dE, energy_pulse_mev, order)

    n_pulse = mask.sum()
    n_tbased = (t_based & mask).sum()
    n_jitter = (jitter & mask).sum()
    n_both = (t_based & jitter & mask).sum()

    return dict(shot=shot, t=t, t_ns=t_ns, v=v, v_c=v_c, xr=xr, pr=pr,
                unreliable=unreliable, t_pulse=t_pulse, energy_mev=energy_pulse_mev,
                order=order, dN_dE=dN_dE, total_protons=total, frac_excluded=frac_excluded,
                n_pulse=n_pulse, pct_tbased_only=100*(n_tbased-n_both)/n_pulse,
                pct_jitter_only=100*(n_jitter-n_both)/n_pulse, pct_both=100*n_both/n_pulse)


def plot_pipeline_walkthrough(shot, V_bias, tag=""):
    r = compute_flux(shot, V_bias)
    t_ns, v, v_c, xr, pr = r["t_ns"], r["v"], r["v_c"], r["xr"], r["pr"]
    print(f"shot {shot}: total protons = {r['total_protons']:.3e}  "
          f"(excluded {r['frac_excluded']*100:.1f}% of pulse samples as unreliable)")

    # wide window: well before the x-ray onset through well after the
    # proton pulse ends, not just the proton pulse itself -- shows the
    # x-ray spike's own exclusions too, which a tight crop hides
    plot_lo, plot_hi = xr["onset_time_ns"] - 20, pr["end_time_ns"] + 20
    plot_mask = (t_ns >= plot_lo) & (t_ns <= plot_hi)

    fig, axes = plt.subplots(2, 2, figsize=(15, 9.5))

    ax = axes[0, 0]
    ax.plot(t_ns[plot_mask], v[plot_mask]*1000, lw=0.7, color="tab:blue")
    ax.axvline(xr["onset_time_ns"], color="red", ls="--", lw=1, label="x-ray onset")
    ax.axvline(pr["onset_time_ns"], color="red", ls=":", lw=1, label="proton onset")
    ax.axvline(pr["end_time_ns"], color="gray", ls=":", lw=1, label="proton end")
    ax.set_title("1. Raw signal (Channel 4)")
    ax.set_xlabel("time (ns)"); ax.set_ylabel("photodiode (mV)"); ax.legend(fontsize=8)
    ax.set_xlim(plot_lo, plot_hi)

    ax = axes[0, 1]
    ax.plot(t_ns[plot_mask], v_c[plot_mask]*1000, lw=0.7, color="tab:green")
    for s, e in regions_above(r["unreliable"]):
        if t_ns[e-1] < plot_lo or t_ns[s] > plot_hi:
            continue
        ax.axvspan(t_ns[s], t_ns[e-1], color="gray", alpha=0.3)
    ax.set_title("2. Pockels-corrected signal (gray = excluded/unreliable)")
    ax.set_xlabel("time (ns)"); ax.set_ylabel("recovered signal (mV)")
    ax.set_xlim(plot_lo, plot_hi)

    ax = axes[1, 0]
    ax.plot(r["t_pulse"], r["energy_mev"], lw=1.2, color="tab:purple")
    ax.set_title("3. Energy axis (from TOF, per sample)")
    ax.set_xlabel("time (ns)"); ax.set_ylabel("proton energy (MeV)")

    ax = axes[1, 1]
    ax.plot(r["energy_mev"][r["order"]], r["dN_dE"][r["order"]],
            marker=".", markersize=2, lw=0.6, color="tab:red")
    ax.set_title(f"4. Flux spectrum: total={r['total_protons']:.2e} protons, "
                 f"{r['frac_excluded']*100:.1f}% excluded")
    ax.set_xlabel("proton energy (MeV)"); ax.set_ylabel("dN/dE (protons/MeV)")

    plt.suptitle(f"Shot {shot} -- full pipeline walkthrough{tag}", fontsize=13)
    plt.tight_layout()
    plt.show()
    return r


_ = plot_pipeline_walkthrough(19, 1.6, " (cleanest of the 4)")


## Every shot with both a Pockels correction and a known flight distance

In [ ]:
FLUX_SHOTS = {19: 1.6, 20: 1.6, 21: 1.1, 27: 2.2}
NO_FLIGHT_DISTANCE = [9, 11]  # have the Pockels correction, but L unknown

results = []
for shot, V_bias in FLUX_SHOTS.items():
    if shot == 19:
        results.append(compute_flux(19, 1.6))  # already plotted above
        continue
    results.append(plot_pipeline_walkthrough(shot, V_bias))


## Summary tables

In [ ]:
rows = []
for shot, V_bias in FLUX_SHOTS.items():
    info = sl.SHOT_LOG[shot]
    r = [x for x in results if x["shot"] == shot][0] if any(x["shot"]==shot for x in results) else compute_flux(shot, V_bias)
    rows.append(dict(
        shot=shot, target=info["target"], total_protons=f"{r['total_protons']:.3e}",
        pct_excluded=round(r["frac_excluded"]*100, 1),
    ))

summary_df = pd.DataFrame(rows).set_index("shot")
summary_df


### Exclusion breakdown by criterion

Which of the two exclusion criteria (pockels.py's T-based mask vs. the
trend-deviation jitter check) actually caught each excluded sample.

In [ ]:
breakdown_rows = []
for shot, V_bias in FLUX_SHOTS.items():
    r = [x for x in results if x["shot"] == shot]
    r = r[0] if r else compute_flux(shot, V_bias)
    breakdown_rows.append(dict(
        shot=shot, n_samples_in_pulse=r["n_pulse"],
        pct_Tbased_only=round(r["pct_tbased_only"], 2),
        pct_jitter_only=round(r["pct_jitter_only"], 2),
        pct_both_criteria=round(r["pct_both"], 2),
        pct_total_excluded=round(r["frac_excluded"]*100, 2),
    ))

breakdown_df = pd.DataFrame(breakdown_rows).set_index("shot")
breakdown_df


## Result

All 4 shots give physically sensible, non-negative flux spectra with
consistent order-of-magnitude totals (~1.0-1.8e5 protons). Two things had to
be fixed to get here, both real physics/data-quality issues rather than
implementation bugs:

1. **Sign**: `v_signal`'s sign is an artifact of which side of `phi_bias`
   the real pulse happens to sit on, not a physical property -- a real
   pulse can come out consistently negative (~98% of samples, confirmed
   this session). A particle flux is inherently non-negative, so the flux
   calculation uses `|v_signal|`, not the signed value.

2. **Peak-region amplitude**: near the transmission curve's flat top,
   `dT/dphi -> 0` is a genuine mathematical singularity -- any residual
   noise, however small, is amplified without bound. No threshold or
   pre-smoothing fixes this (checked directly this session). Those samples
   are flagged (via a T-based check plus a separate trend-deviation jitter
   check, since large-but-real amplitude near a genuine peak isn't itself
   suspicious) and excluded as NaN -- an honest "unknown," not a guess.
   The breakdown table above shows the jitter check does essentially all
   of the real protective work (T-based-only is 0.0% on every shot) --
   the original threshold, tuned for timing, isn't tight enough for flux
   on its own.

Shots 9 and 11 have the Pockels correction but no known flight distance yet,
so no energy axis -- flux isn't computable for them until that's resolved.